# Babel Currency Probe

This notebook is a small probe for `babel.numbers.get_territory_currencies(...)`.

Goals:
- verify whether Babel is installed in the local environment
- inspect current tender currency outputs for a representative set of territories
- compare Babel outputs against the current fallback country-to-currency map used in analysis
- inspect historical behavior for territories where current and historical tender currencies may differ


In [ ]:
import sys
import os
import importlib.util
from datetime import date

import pandas as pd
import pycountry

# Add project root to sys.path if running from within the notebooks directory
if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.append(os.path.abspath(".."))

try:
    from babel.numbers import get_territory_currencies
    BABEL_AVAILABLE = True
except ImportError:
    get_territory_currencies = None
    BABEL_AVAILABLE = False

import pystocks.analysis as analysis_module

fallback_map = getattr(analysis_module, "COUNTRY_CURRENCY_MAP", {})

print({
    "babel_installed": BABEL_AVAILABLE,
    "pycountry_installed": importlib.util.find_spec("pycountry") is not None,
    "fallback_map_entries": len(fallback_map),
})

In [ ]:
territory_cases = [
    {"label": "United States", "alpha2": "US", "alpha3": "USA"},
    {"label": "Canada", "alpha2": "CA", "alpha3": "CAN"},
    {"label": "United Kingdom", "alpha2": "GB", "alpha3": "GBR"},
    {"label": "Japan", "alpha2": "JP", "alpha3": "JPN"},
    {"label": "China", "alpha2": "CN", "alpha3": "CHN"},
    {"label": "Hong Kong", "alpha2": "HK", "alpha3": "HKG"},
    {"label": "Taiwan", "alpha2": "TW", "alpha3": "TWN"},
    {"label": "Korea", "alpha2": "KR", "alpha3": "KOR"},
    {"label": "Switzerland", "alpha2": "CH", "alpha3": "CHE"},
    {"label": "Norway", "alpha2": "NO", "alpha3": "NOR"},
    {"label": "Singapore", "alpha2": "SG", "alpha3": "SGP"},
    {"label": "Australia", "alpha2": "AU", "alpha3": "AUS"},
    {"label": "New Zealand", "alpha2": "NZ", "alpha3": "NZL"},
    {"label": "South Africa", "alpha2": "ZA", "alpha3": "ZAF"},
    {"label": "Mexico", "alpha2": "MX", "alpha3": "MEX"},
    {"label": "Germany", "alpha2": "DE", "alpha3": "DEU"},
    {"label": "France", "alpha2": "FR", "alpha3": "FRA"},
    {"label": "Spain", "alpha2": "ES", "alpha3": "ESP"},
    {"label": "Italy", "alpha2": "IT", "alpha3": "ITA"},
    {"label": "Luxembourg", "alpha2": "LU", "alpha3": None},
    {"label": "Ireland", "alpha2": "IE", "alpha3": None},
    {"label": "Jersey", "alpha2": "JE", "alpha3": None},
    {"label": "Guernsey", "alpha2": "GG", "alpha3": None},
]


def probe_territory_currency(case):
    alpha2 = case["alpha2"]
    alpha3 = case["alpha3"]
    fallback_currency = fallback_map.get(alpha3.lower()) if alpha3 else None
    fallback_currency_upper = fallback_currency.upper() if fallback_currency else None
    if not BABEL_AVAILABLE:
        babel_current = None
        babel_details = None
    else:
        babel_current = get_territory_currencies(alpha2, tender=True)
        babel_details = get_territory_currencies(
            alpha2,
            tender=True,
            include_details=True,
        )
    babel_primary = babel_current[-1] if babel_current else None
    return {
        "label": case["label"],
        "alpha2": alpha2,
        "alpha3": alpha3,
        "fallback_currency": fallback_currency,
        "fallback_currency_upper": fallback_currency_upper,
        "babel_current": babel_current,
        "babel_primary": babel_primary,
        "matches_fallback": (
            None
            if fallback_currency_upper is None or babel_primary is None
            else fallback_currency_upper == babel_primary
        ),
        "babel_details": babel_details,
    }


probe_df = pd.DataFrame([probe_territory_currency(case) for case in territory_cases])
probe_df[[
    "label",
    "alpha2",
    "alpha3",
    "fallback_currency",
    "fallback_currency_upper",
    "babel_current",
    "babel_primary",
    "matches_fallback",
]]

In [ ]:
if BABEL_AVAILABLE:
    mismatches = probe_df.loc[
        probe_df["matches_fallback"].eq(False),
        ["label", "alpha2", "alpha3", "fallback_currency", "babel_current", "babel_details"],
    ]
    no_primary = probe_df.loc[
        probe_df["babel_primary"].isna(),
        ["label", "alpha2", "alpha3", "babel_current", "babel_details"],
    ]

    print("Mismatches against fallback map:")
    display(mismatches if not mismatches.empty else pd.DataFrame({"status": ["none"]}))

    print("Territories with no primary current tender currency returned:")
    display(no_primary if not no_primary.empty else pd.DataFrame({"status": ["none"]}))
else:
    print("Babel is not installed. To run the probe, install it in the venv first.")

In [ ]:
historical_cases = ["DE", "FR", "ES", "IT", "LU", "IE", "GB", "JE", "GG"]
historical_dates = [
    date(1990, 1, 1),
    date(2000, 1, 1),
    date(2010, 1, 1),
    date.today(),
]

if BABEL_AVAILABLE:
    historical_rows = []
    for territory in historical_cases:
        country = pycountry.countries.get(alpha_2=territory)
        label = country.name if country is not None else territory
        for probe_date in historical_dates:
            historical_rows.append({
                "territory": territory,
                "label": label,
                "probe_date": probe_date.isoformat(),
                "currencies": get_territory_currencies(
                    territory,
                    start_date=probe_date,
                    end_date=probe_date,
                    tender=True,
                ),
            })
    historical_df = pd.DataFrame(historical_rows)
    historical_df
else:
    print("Babel is not installed. Historical probe skipped.")